# Table S7: Ribosome density and spacing

The initiation and elongation rates are the medians of the 320 lowest-cost ACF grid points used for Fig. S11. Brackets show the 5th–95th percentiles of those points, not confidence intervals. Calculations use unrounded rates.

In [1]:
from pathlib import Path

import pandas as pd
from tasep_models import read_sequence, tag_dict

working_dir = Path.cwd().resolve()
main_dir = next(p for p in (working_dir, *working_dir.parents)
                if (p / 'src').is_dir() and (p / 'notebooks').is_dir())
results_dir = main_dir / 'optimization' / 'results_ACF'
sequence_dir = main_dir / 'gene_sequences' / 'utag_project'

In [2]:
utag_reporter = 'pNZ208(pUB-24xUTagFullLength-KDM5B-MS2).dna'
reporters = {
    'utag': ('Anti-UTag-IB', 'U', utag_reporter),
    'utag_c_free': ('Anti-UTag-IB(ΔCys)', 'U', utag_reporter),
    'suntag': ('Anti-SunTag-IB', 'SUN', 'pNZ266(pUB-24xGCN4-KDM5B-MS2).dna'),
    'alfatag': ('Anti-ALFA-IB', 'ALFA', 'pNZ267 (pUB-24xALFAtag-KDM5B-MS2).dna'),
}
footprint_codons = 9

In [3]:
def format_rate(params, rate, decimals):
    """Format the saved median and percentile range for display in Table S7."""
    low = params[f'{rate}_90%_CI_min']
    high = params[f'{rate}_90%_CI_max']
    return f'{params[rate]:.{decimals}f} [{low:.{decimals}f}, {high:.{decimals}f}]'

In [4]:
rows = []
for dataset, (label, tag, filename) in reporters.items():
    params = pd.read_csv(results_dir / f'optimized_params_{dataset}.csv', float_precision='round_trip')
    params = params.set_index('parameter', verify_integrity=True)['value']
    ki, ke = params['ki'], params['ke']
    protein, *_ = read_sequence(seq=sequence_dir / filename, min_protein_length=50, TAG=[tag_dict[tag]])
    length_codons = len(protein) + 1  # Include the stop codon, as in the ACF model.

    rows.append({
        'Tagging system': label,
        'ki (s⁻¹)': format_rate(params, 'ki', 4),
        'ke (aa/s)': format_rate(params, 'ke', 2),
        'Coding length (codons)': length_codons,
        'Density (%)': 100 * footprint_codons * ki / ke,
        'Ribosomes/mRNA': ki * length_codons / ke,
        'Spacing (codons)': ke / ki,
    })

table_s7 = pd.DataFrame(rows)

In [5]:
print(table_s7.to_string(index=False, float_format=lambda value: f'{value:.1f}'))

    Tagging system                ki (s⁻¹)         ke (aa/s)  Coding length (codons)  Density (%)  Ribosomes/mRNA  Spacing (codons)
      Anti-UTag-IB 0.0317 [0.0171, 0.0472] 3.11 [2.10, 4.73]                    1968          9.2            20.0              98.2
Anti-UTag-IB(ΔCys) 0.0636 [0.0516, 0.0765] 5.34 [4.13, 6.66]                    1968         10.7            23.4              84.0
    Anti-SunTag-IB 0.0295 [0.0171, 0.0410] 4.33 [2.81, 5.95]                    2133          6.1            14.5             146.8
      Anti-ALFA-IB 0.0437 [0.0330, 0.0543] 4.84 [3.92, 5.95]                    2016          8.1            18.2             110.7
